# THANTD — Colab Training Notebook
**Triplet Hybrid Attention Network for Hyperspectral Target Detection**  
Liu et al., IEEE JSTARS 2025

### What this notebook does
1. Mounts your Google Drive  
2. Installs the `hyspan` library from GitHub  
3. Loads 2–3 real HSI datasets from `MyDrive/hyspan_datasets/`  
4. Per dataset: synthesises a **(500 × 250)** HSI using GMM-based synthesis  
5. Splits into **train (350 × 250)** and **test (150 × 250)**  
6. Trains THANTD on the training split  
7. Evaluates on the test split (AUC + detection map)  
8. Saves synthetic images, checkpoints, and results to Drive

### Drive layout expected
```
MyDrive/
  hyspan_datasets/
    Sandiego.mat          # key: 'data' (H,W,D), 'map' (H,W)
    abu-airport-2.mat
    (pavia-u.mat)         # optional — very large
```
Each `.mat` file should contain at minimum a `data` field with the HSI array
and a `map` field with the binary ground-truth (1=target, 0=background).

## 0 · Runtime check

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU.')

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT    = '/content/drive/MyDrive'
DATASET_DIR   = os.path.join(DRIVE_ROOT, 'hyspan_datasets')
OUTPUT_DIR    = os.path.join(DRIVE_ROOT, 'hyspan_outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Datasets:', os.listdir(DATASET_DIR))

## 2 · Install hyspan

In [ ]:
!pip install -q git+https://github.com/michaelpiro/hyspan.git
# Reload site-packages in case of first install
import importlib, site
importlib.invalidate_caches()

## 3 · Imports

In [ ]:
import os
import math
import json
import numpy as np
import scipy.io as sio
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from hyspan.deep_models.THANTD import (
    THANTD, ETBLoss, TripletDataset,
    build_dataset_from_image, train_thantd, thantd_detect,
)
from hyspan.synthesis     import SynthesisEngine
from hyspan.results       import roc_auc
from hyspan.algorithms.utils import ts_generation

print('hyspan imported OK')

## 4 · Configuration

In [ ]:
# ── Dataset files to process (must exist in DATASET_DIR) ──────────────────
DATASET_FILES = [
    'Sandiego.mat',
    'abu-airport-2.mat',
    # 'pavia-u.mat',   # uncomment if you have it — very large, needs lots of RAM
]

# ── Synthesis settings ────────────────────────────────────────────────────
SYNTH_H        = 500    # full synthetic image height
SYNTH_W        = 250    # full synthetic image width
TRAIN_H        = 350    # rows kept for training
TEST_H         = 150    # rows kept for testing  (TRAIN_H + TEST_H == SYNTH_H)
N_GMM_COMPS    = 10     # GMM components for synthesis
PCA_COMPONENTS = 30     # reduce to this many bands before training
                        # (set to None to skip PCA)
N_TRIPLETS     = 60_000 # training triplets per epoch

# ── THANTD model ──────────────────────────────────────────────────────────
MODEL_CFG = dict(
    d_model       = 64,
    n_heads       = 4,
    n_layers      = 2,
    window_size   = 3,
    mlp_ratio     = 4,
    cam_reduction = 4,
    dropout       = 0.1,
)

# ── Training (only args accepted by train_thantd) ─────────────────────────
TRAIN_CFG = dict(
    epochs       = 60,
    batch_size   = 512,
    lr           = 1e-4,
    weight_decay = 1e-4,
    margin       = 0.3,
    lambda_bce   = 0.5,
    device       = device,
    print_every  = 10,
)

## 5 · Helper utilities

In [ ]:
def load_mat(path):
    """Load a .mat file and return (image, gt) as float32 numpy arrays."""
    mat  = sio.loadmat(path)
    # Try common key names for the HSI data
    for key in ['data', 'Data', 'img', 'image', 'HSI', 'hsi']:
        if key in mat:
            img = mat[key].astype(np.float32)
            break
    else:
        keys = [k for k in mat if not k.startswith('_')]
        img  = mat[keys[0]].astype(np.float32)
    # Try common key names for ground truth
    gt = None
    for key in ['map', 'Map', 'gt', 'GT', 'groundtruth', 'label']:
        if key in mat:
            gt = mat[key].astype(np.float32)
            break
    if img.ndim == 2:                         # (H, W*D) packed
        raise ValueError('2D mat not supported — check array layout')
    if img.shape[0] < img.shape[-1]:          # (D, H, W) → (H, W, D)
        img = img.transpose(1, 2, 0)
    # Normalise per-band to [0, 1]
    mn, mx = img.min(axis=(0,1), keepdims=True), img.max(axis=(0,1), keepdims=True)
    img = (img - mn) / (mx - mn + 1e-8)
    print(f'  image: {img.shape}, gt: {gt.shape if gt is not None else None}')
    return img, gt


class TorchPCA:
    """Pure-torch PCA — avoids sklearn/numpy ABI issues."""
    def __init__(self, n_components):
        self.k = n_components
        self._components = None
        self._mean       = None

    def fit(self, pixels: torch.Tensor):            # (N, D)
        self._mean = pixels.mean(dim=0)             # (D,)
        X = pixels - self._mean
        cov = X.T @ X / (X.shape[0] - 1)           # (D, D)
        _, vecs = torch.linalg.eigh(cov)            # ascending eigenvalues
        self._components = vecs[:, -self.k:].flip(-1)  # (D, k) top-k
        return self

    def transform(self, image: torch.Tensor) -> torch.Tensor:   # (H, W, D)
        H, W, D = image.shape
        x = image.reshape(-1, D) - self._mean
        return (x @ self._components).reshape(H, W, self.k)

    def transform_vector(self, v: torch.Tensor) -> torch.Tensor:  # (D,)
        return (v - self._mean) @ self._components  # (k,)


def to_tensor(arr):
    """numpy array → torch float32 tensor, safe on NumPy 2.x."""
    return torch.tensor(arr.tolist(), dtype=torch.float32)


def plot_results(train_image, test_image, test_gt, scores, history, title, save_path=None):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(title, fontsize=13)

    # False-colour composite of test image (bands 0,1,2)
    rgb = test_image[:, :, :3].numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    axes[0].imshow(rgb); axes[0].set_title('Test image (RGB)'); axes[0].axis('off')

    # Ground truth
    axes[1].imshow(test_gt.numpy(), cmap='gray')
    axes[1].set_title('Ground truth'); axes[1].axis('off')

    # Detection map
    axes[2].imshow(scores.numpy(), cmap='hot')
    axes[2].set_title('THANTD scores'); axes[2].axis('off')

    # Training loss curve
    epochs     = [m['epoch'] for m in history]
    total_loss = [m['loss']  for m in history]
    dual_loss  = [m['loss_dual'] for m in history]
    axes[3].plot(epochs, total_loss, label='Total')
    axes[3].plot(epochs, dual_loss,  label='Dual triplet', linestyle='--')
    axes[3].set_xlabel('Epoch'); axes[3].set_ylabel('Loss')
    axes[3].set_title('Training loss'); axes[3].legend()
    axes[3].grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved plot → {save_path}')
    plt.show()


print('Utilities ready.')

## 6 · Main loop: synthesise → train → evaluate

In [ ]:
def synthesise_with_targets(engine, image_real, gt_real, H, W, seed=42):
    D = image_real.shape[-1]
    synth_bg = engine.generate(H, W, seed=seed)
    if gt_real is None or gt_real.sum() == 0:
        raise RuntimeError('gt has no target pixels — cannot plant targets')
    tgt_px   = image_real.reshape(-1, D)[gt_real.reshape(-1) == 1]
    tgt_mean = tgt_px.mean(dim=0)
    tgt_std  = tgt_px.std(dim=0).clamp(min=0)
    torch.manual_seed(seed)
    n_targets = max(20, int(0.005 * H * W))
    target_hs = torch.randint(0, H, (n_targets,))
    target_ws = torch.randint(0, W, (n_targets,))
    synth_image = synth_bg.clone()
    synth_gt    = torch.zeros(H, W, dtype=torch.float32)
    for h, w in zip(target_hs.tolist(), target_ws.tolist()):
        noise = torch.randn(D) * tgt_std * 0.1
        synth_image[h, w] = (tgt_mean + noise).clamp(0, 1)
        synth_gt[h, w]    = 1.0
    return synth_image, synth_gt


def ensure_hwD(t: torch.Tensor) -> torch.Tensor:
    """
    Guarantee a 3-D tensor is (H, W, D) — spectral axis last.

    Strategy: when exactly two dimensions are equal, the third is spectral.
    Handles all common HSI storage layouts:
      (H, W, D) — already correct
      (H, D, W) — two equal spatial dims at axis 0 and 2
      (D, H, W) — two equal spatial dims at axis 1 and 2
    Falls back to largest-dim = spectral when all three sizes differ.
    """
    if t.ndim != 3:
        return t
    a, b, c = t.shape
    if a == b and a != c:   return t                              # (H, W, D) ✓
    if a == c and a != b:   return t.permute(0, 2, 1).contiguous()  # (H, D, W) → (H, W, D)
    if b == c and b != a:   return t.permute(1, 2, 0).contiguous()  # (D, H, W) → (H, W, D)
    # All three differ — put the axis with the value most unlike the other two last
    # (works for non-square images with different H, W, D)
    dims = [a, b, c]
    diffs = [abs(dims[i] - dims[(i+1)%3]) + abs(dims[i] - dims[(i+2)%3]) for i in range(3)]
    d_axis = int(torch.tensor(diffs).argmax().item())
    if d_axis == 0: return t.permute(1, 2, 0).contiguous()
    if d_axis == 1: return t.permute(0, 2, 1).contiguous()
    return t


all_results = {}

for mat_file in DATASET_FILES:
    dataset_name = mat_file.replace('.mat', '')
    print(f'\n{"="*60}')
    print(f'Dataset: {dataset_name}')
    print('='*60)

    # ── 6.1 Load real dataset ─────────────────────────────────────────────
    mat_path      = os.path.join(DATASET_DIR, mat_file)
    img_np, gt_np = load_mat(mat_path)
    image_real    = ensure_hwD(to_tensor(img_np))
    gt_real       = to_tensor(gt_np) if gt_np is not None else None
    D             = image_real.shape[-1]
    print(f'  Loaded: {image_real.shape}, bands={D}')

    # ── 6.2 Synthesise (500 × 250) image ──────────────────────────────────
    print('Fitting SynthesisEngine to real dataset …')
    engine = SynthesisEngine.from_image(
        image_real, n_components=N_GMM_COMPS, gt=gt_real, device=device,
    )
    print('Generating synthetic image (500 × 250) …')
    synth_image, synth_gt = synthesise_with_targets(
        engine, image_real, gt_real, SYNTH_H, SYNTH_W, seed=42
    )
    print(f'  Synthetic image: {synth_image.shape},  targets: {synth_gt.sum().item():.0f} pixels')

    # ── 6.3 Save synthetic images to Drive ────────────────────────────────
    synth_dir = os.path.join(OUTPUT_DIR, dataset_name)
    os.makedirs(synth_dir, exist_ok=True)
    torch.save({'image': synth_image, 'gt': synth_gt}, os.path.join(synth_dir, 'synthetic_full.pt'))
    print(f'  Saved → {synth_dir}/synthetic_full.pt')

    # ── 6.4 Train / test split ────────────────────────────────────────────
    train_image = synth_image[:TRAIN_H]
    train_gt    = synth_gt[:TRAIN_H]
    test_image  = synth_image[TRAIN_H:]
    test_gt     = synth_gt[TRAIN_H:]
    torch.save({'image': train_image, 'gt': train_gt}, os.path.join(synth_dir, 'train.pt'))
    torch.save({'image': test_image,  'gt': test_gt},  os.path.join(synth_dir, 'test.pt'))
    print(f'  Train targets: {train_gt.sum().item():.0f}   Test targets: {test_gt.sum().item():.0f}')

    # ── 6.5 Optional PCA ──────────────────────────────────────────────────
    n_bands = D
    pca     = None
    if PCA_COMPONENTS is not None and D > PCA_COMPONENTS:
        print(f'Applying PCA: {D} → {PCA_COMPONENTS} bands …')
        pca = TorchPCA(PCA_COMPONENTS)
        pca.fit(train_image.reshape(-1, D))
        train_image = pca.transform(train_image)
        test_image  = pca.transform(test_image)
        n_bands     = PCA_COMPONENTS
        print(f'  PCA done: train {train_image.shape}, test {test_image.shape}')

    # ── 6.6 Target spectrum ────────────────────────────────────────────────
    tgt_pixels = train_image.reshape(-1, n_bands)[train_gt.reshape(-1) == 1]
    if tgt_pixels.shape[0] == 0:
        raise RuntimeError('No target pixels in training split!')
    prior = tgt_pixels.mean(dim=0)
    print(f'  Prior spectrum from {tgt_pixels.shape[0]} target pixels')

    # ── 6.7 TripletDataset ────────────────────────────────────────────────
    print('Building TripletDataset …')
    trip_dataset = build_dataset_from_image(
        train_image, train_gt, prior, n_triplets=N_TRIPLETS, mu_max=0.1,
    )
    print(f'  Dataset size: {len(trip_dataset)} triplets')

    # ── 6.8 Train THANTD ──────────────────────────────────────────────────
    model    = THANTD(n_bands=n_bands, **MODEL_CFG)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'THANTD  |  bands={n_bands}  d={MODEL_CFG["d_model"]}  '
          f'layers={MODEL_CFG["n_layers"]}  params={n_params:,}')

    ckpt_path = os.path.join(synth_dir, 'thantd_best.pt')
    print('Training …')
    history = train_thantd(model, trip_dataset, checkpoint_path=ckpt_path, **TRAIN_CFG)
    print(f'Training done. Best checkpoint → {ckpt_path}')
    torch.save(model.state_dict(), os.path.join(synth_dir, 'thantd_final.pt'))

    # ── 6.9 Evaluate on synthetic test split ──────────────────────────────
    print('Evaluating on test split …')
    scores = thantd_detect(model, test_image.to(device), prior.to(device), normalise=True)
    auc    = roc_auc(scores.reshape(-1), test_gt.reshape(-1))
    print(f'  AUC = {auc:.4f}')

    # ── 6.10 Save ─────────────────────────────────────────────────────────
    torch.save(scores, os.path.join(synth_dir, 'test_scores.pt'))
    with open(os.path.join(synth_dir, 'history.json'), 'w') as f:
        json.dump(history, f, indent=2)
    all_results[dataset_name] = {'auc': auc, 'history': history}

    # ── 6.11 Plot ─────────────────────────────────────────────────────────
    plot_results(
        train_image.cpu(), test_image.cpu(), test_gt.cpu(), scores.cpu(),
        history,
        title=f'THANTD — {dataset_name}  (AUC={auc:.4f})',
        save_path=os.path.join(synth_dir, 'results.png'),
    )

print('\n' + '='*60)
print('ALL DONE')
for name, res in all_results.items():
    print(f'  {name:25s}  AUC = {res["auc"]:.4f}')

## 7 · Inspect a single result in more detail

In [ ]:
# Change this to whichever dataset you want to inspect
INSPECT = 'Sandiego'

synth_dir   = os.path.join(OUTPUT_DIR, INSPECT)
test_data   = torch.load(os.path.join(synth_dir, 'test.pt'))
test_scores = torch.load(os.path.join(synth_dir, 'test_scores.pt'))
history     = json.load(open(os.path.join(synth_dir, 'history.json')))

test_image_v = test_data['image']
test_gt_v    = test_data['gt']

# ROC curve
scores_flat = test_scores.reshape(-1)
gt_flat     = test_gt_v.reshape(-1)
thresholds  = torch.linspace(0, 1, 200)
pd_list, pf_list = [], []
for tau in thresholds:
    pred = (scores_flat >= tau).float()
    tp   = (pred * gt_flat).sum()
    fp   = (pred * (1 - gt_flat)).sum()
    fn   = ((1 - pred) * gt_flat).sum()
    tn   = ((1 - pred) * (1 - gt_flat)).sum()
    pd_list.append((tp / (tp + fn + 1e-8)).item())
    pf_list.append((fp / (fp + tn + 1e-8)).item())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'THANTD — {INSPECT}', fontsize=13)

axes[0].plot(pf_list, pd_list)
axes[0].plot([0,1],[0,1],'k--', alpha=0.4)
axes[0].set_xlabel('False Alarm Rate'); axes[0].set_ylabel('Detection Rate')
axes[0].set_title(f'ROC  (AUC={all_results[INSPECT]["auc"]:.4f})')
axes[0].grid(True, alpha=0.3)

axes[1].imshow(test_scores.numpy(), cmap='hot')
axes[1].set_title('Detection scores'); axes[1].axis('off')

axes[2].imshow(test_gt_v.numpy(), cmap='gray')
axes[2].set_title('Ground truth'); axes[2].axis('off')

plt.tight_layout(); plt.show()

## 8 · Load a saved checkpoint and re-run inference

In [ ]:
"""
Distribution analysis: how well does the synthetic data represent the real data?

Metrics:
  1. Per-band mean ± std profiles  (real vs synthetic background)
  2. PCA scatter PC1 vs PC2        (real vs synthetic pixel clouds)
  3. MMD² (Maximum Mean Discrepancy) — scalar distribution distance
  4. Per-band histogram IoU        (marginal overlap per band)
"""
import matplotlib.gridspec as gridspec

DIST_DATASET = 'Sandiego'   # change to any dataset in all_results

synth_dir = os.path.join(OUTPUT_DIR, DIST_DATASET)
mat_path  = os.path.join(DATASET_DIR, DIST_DATASET + '.mat')

# ── Load and fix layout ────────────────────────────────────────────────────
img_np_r, gt_np_r = load_mat(mat_path)
real_img  = ensure_hwD(to_tensor(img_np_r))   # (H, W, D)
real_gt   = to_tensor(gt_np_r)

synth_full = torch.load(os.path.join(synth_dir, 'synthetic_full.pt'), map_location='cpu')
synth_img  = synth_full['image']   # (500, 250, D)
synth_gt   = synth_full['gt']

D = real_img.shape[-1]
assert real_img.shape[-1] == synth_img.shape[-1], \
    f'Band mismatch: real={real_img.shape}, synth={synth_img.shape}'

# Flatten to pixel sets — background only
real_bg_px  = real_img.reshape(-1, D)[real_gt.reshape(-1)  == 0]
synth_bg_px = synth_img.reshape(-1, D)[synth_gt.reshape(-1) == 0]
real_tg_px  = real_img.reshape(-1, D)[real_gt.reshape(-1)  == 1]
synth_tg_px = synth_img.reshape(-1, D)[synth_gt.reshape(-1) == 1]

print(f'Real  background: {real_bg_px.shape[0]:,} px   target: {real_tg_px.shape[0]:,} px')
print(f'Synth background: {synth_bg_px.shape[0]:,} px   target: {synth_tg_px.shape[0]:,} px')

def subsample(x, n=5000, seed=0):
    torch.manual_seed(seed)
    idx = torch.randperm(x.shape[0])[:min(n, x.shape[0])]
    return x[idx]

real_sub  = subsample(real_bg_px,  5000)
synth_sub = subsample(synth_bg_px, 5000)

# ── 1. Per-band mean ± std ─────────────────────────────────────────────────
bands = np.arange(D)
r_mean, r_std = real_bg_px.mean(0).numpy(),  real_bg_px.std(0).numpy()
s_mean, s_std = synth_bg_px.mean(0).numpy(), synth_bg_px.std(0).numpy()
mae = np.abs(r_mean - s_mean)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
fig.suptitle(f'Distribution Analysis — {DIST_DATASET}', fontsize=13)
ax = axes[0]
ax.plot(bands, r_mean, 'b',   label='Real mean')
ax.fill_between(bands, r_mean-r_std, r_mean+r_std, alpha=0.2, color='blue')
ax.plot(bands, s_mean, 'r--', label='Synth mean')
ax.fill_between(bands, s_mean-s_std, s_mean+s_std, alpha=0.2, color='red')
ax.set_xlabel('Band index'); ax.set_ylabel('Reflectance [0-1]')
ax.set_title('Per-band mean ± std (background)'); ax.legend(); ax.grid(True, alpha=0.3)
axes[1].bar(bands, mae, color='purple', alpha=0.7)
axes[1].set_xlabel('Band index'); axes[1].set_ylabel('|mean_real − mean_synth|')
axes[1].set_title(f'Per-band mean MAE  (avg={mae.mean():.4f})'); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(synth_dir, 'dist_band_profiles.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── 2. PCA scatter ────────────────────────────────────────────────────────
all_px = torch.cat([real_sub, synth_sub], dim=0)
pca2   = TorchPCA(2)
pca2.fit(all_px)
r_2d = ((real_sub  - pca2._mean) @ pca2._components).numpy()
s_2d = ((synth_sub - pca2._mean) @ pca2._components).numpy()

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(r_2d[:,0], r_2d[:,1], s=5, alpha=0.4, color='blue',   label='Real background')
ax.scatter(s_2d[:,0], s_2d[:,1], s=5, alpha=0.4, color='red',    label='Synth background')
if real_tg_px.shape[0] > 0:
    rt_2d = ((real_tg_px - pca2._mean) @ pca2._components).numpy()
    ax.scatter(rt_2d[:,0], rt_2d[:,1], s=30, marker='*', color='cyan', label='Real target', zorder=5)
if synth_tg_px.shape[0] > 0:
    st_2d = ((subsample(synth_tg_px, 200) - pca2._mean) @ pca2._components).numpy()
    ax.scatter(st_2d[:,0], st_2d[:,1], s=20, marker='^', color='orange', label='Synth target', zorder=5)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title(f'PCA scatter — {DIST_DATASET}'); ax.legend(markerscale=3); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(synth_dir, 'dist_pca_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── 3. MMD² with RBF kernel ───────────────────────────────────────────────
def rbf_mmd(X, Y, n=2000):
    X, Y = subsample(X, n).float(), subsample(Y, n).float()
    all_xy = torch.cat([X, Y])
    sigma  = torch.cdist(all_xy, all_xy).median().item() + 1e-8
    gamma  = 1.0 / (2 * sigma**2)
    Kxx = torch.exp(-gamma * torch.cdist(X, X)**2).mean()
    Kyy = torch.exp(-gamma * torch.cdist(Y, Y)**2).mean()
    Kxy = torch.exp(-gamma * torch.cdist(X, Y)**2).mean()
    return (Kxx + Kyy - 2*Kxy).item()

mmd_bg  = rbf_mmd(real_bg_px, synth_bg_px)
mmd_tgt = rbf_mmd(real_tg_px, synth_tg_px) if real_tg_px.shape[0] > 0 and synth_tg_px.shape[0] > 0 else float('nan')
print(f'\nMMD² background: {mmd_bg:.6f}')
print(f'MMD² target:     {mmd_tgt:.6f}  (lower = more similar)')

# ── 4. Per-band histogram IoU ─────────────────────────────────────────────
n_bins       = 50
overlaps     = []
sample_bands = list(range(0, D, max(1, D // 8)))[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle(f'Per-band marginal histograms — {DIST_DATASET}', fontsize=13)
for ax, b in zip(axes.flat, sample_bands):
    bins  = np.linspace(0, 1, n_bins + 1)
    r_h, _ = np.histogram(real_bg_px[:, b].numpy(),  bins=bins, density=True)
    s_h, _ = np.histogram(synth_bg_px[:, b].numpy(), bins=bins, density=True)
    iou    = np.minimum(r_h, s_h).sum() / (np.maximum(r_h, s_h).sum() + 1e-8)
    overlaps.append(iou)
    ax.bar(bins[:-1], r_h, width=1/n_bins, alpha=0.5, color='blue', label='Real')
    ax.bar(bins[:-1], s_h, width=1/n_bins, alpha=0.5, color='red',  label='Synth')
    ax.set_title(f'Band {b}  (IoU={iou:.2f})'); ax.set_xlabel('Value'); ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(synth_dir, 'dist_histograms.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Summary ───────────────────────────────────────────────────────────────
auc_synth = all_results.get(DIST_DATASET, {}).get('auc', float('nan'))
print('\n' + '='*50)
print(f'Distribution summary — {DIST_DATASET}')
print('='*50)
print(f'  Per-band mean MAE:      {mae.mean():.4f}')
print(f'  MMD² background:        {mmd_bg:.6f}')
print(f'  MMD² target:            {mmd_tgt:.6f}')
print(f'  Histogram IoU (avg):    {np.mean(overlaps):.3f}  (1.0 = perfect)')
print(f'  AUC on synthetic test:  {auc_synth:.4f}')
print('='*50)

## 10 · Distribution analysis: real vs synthetic

In [ ]:
# ── Config: which dataset + checkpoint to use ─────────────────────────────
EVAL_DATASET = 'Sandiego'   # change to 'abu-airport-2' etc.

synth_dir  = os.path.join(OUTPUT_DIR, EVAL_DATASET)
mat_path   = os.path.join(DATASET_DIR, EVAL_DATASET + '.mat')

# ── Load real dataset ──────────────────────────────────────────────────────
print(f'Loading real dataset: {EVAL_DATASET}')
img_np_real, gt_np_real = load_mat(mat_path)
image_real_eval = ensure_hwD(to_tensor(img_np_real))   # <-- ensure_hwD fixes layout
gt_real_eval    = to_tensor(gt_np_real)
D_real          = image_real_eval.shape[-1]
print(f'  Real image: {image_real_eval.shape}  targets: {gt_real_eval.sum():.0f}')

# ── Load training data to re-derive PCA + prior ───────────────────────────
train_data   = torch.load(os.path.join(synth_dir, 'train.pt'), map_location='cpu')
train_img    = train_data['image']   # (350, 250, D_orig) — before PCA
train_gt     = train_data['gt']

# Re-fit PCA on the synthetic training pixels (same transform used at train time)
D_orig = train_img.shape[-1]
pca_eval = None
n_bands_eval = D_orig
if PCA_COMPONENTS is not None and D_orig > PCA_COMPONENTS:
    print(f'Re-fitting PCA: {D_orig} → {PCA_COMPONENTS} bands on synthetic train pixels …')
    pca_eval = TorchPCA(PCA_COMPONENTS)
    pca_eval.fit(train_img.reshape(-1, D_orig))
    n_bands_eval = PCA_COMPONENTS

# Prior from synthetic train targets
train_img_pca = pca_eval.transform(train_img) if pca_eval else train_img
prior_eval    = train_img_pca.reshape(-1, n_bands_eval)[train_gt.reshape(-1) == 1].mean(dim=0)

# ── Apply same PCA to real image ───────────────────────────────────────────
if pca_eval is not None:
    image_real_pca = pca_eval.transform(image_real_eval)
else:
    image_real_pca = image_real_eval
print(f'  Real image after PCA: {image_real_pca.shape}')

# ── Load best checkpoint ───────────────────────────────────────────────────
ckpt = torch.load(os.path.join(synth_dir, 'thantd_best.pt'), map_location=device)
model_eval = THANTD(n_bands=n_bands_eval, **MODEL_CFG)
model_eval.load_state_dict(ckpt['model_state'])
model_eval.to(device).eval()
print(f'  Checkpoint loaded from epoch {ckpt["epoch"]} (loss={ckpt["loss"]:.4f})')

# ── Detect on real image ───────────────────────────────────────────────────
scores_real = thantd_detect(model_eval, image_real_pca.to(device), prior_eval.to(device), normalise=True)
auc_real    = roc_auc(scores_real.reshape(-1), gt_real_eval.reshape(-1))
print(f'\n  AUC on REAL dataset ({EVAL_DATASET}): {auc_real:.4f}')

# ── Also get AUC from synthetic test for comparison ────────────────────────
test_data  = torch.load(os.path.join(synth_dir, 'test.pt'), map_location='cpu')
test_scores_synth = torch.load(os.path.join(synth_dir, 'test_scores.pt'), map_location='cpu')
auc_synth  = roc_auc(test_scores_synth.reshape(-1), test_data['gt'].reshape(-1))
print(f'  AUC on SYNTHETIC test:                {auc_synth:.4f}')

# ── Plot: detection map on real image ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'THANTD on REAL {EVAL_DATASET}  (AUC={auc_real:.4f})', fontsize=13)

rgb = image_real_eval[:, :, [30, 15, 5]].numpy()
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
axes[0].imshow(rgb); axes[0].set_title('Real image (false colour)'); axes[0].axis('off')
axes[1].imshow(gt_real_eval.numpy(), cmap='gray'); axes[1].set_title('Ground truth'); axes[1].axis('off')
axes[2].imshow(scores_real.numpy(), cmap='hot'); axes[2].set_title('THANTD scores'); axes[2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(synth_dir, 'real_eval.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {synth_dir}/real_eval.png')

## 9 · Evaluate on real dataset (transfer from synthetic)